## Importovanie knižníc

In [ ]:
import pandas as pd
import matplotlib
import seaborn as sns

## Analýza štruktúr dát ako súbory

Dataset sa skladá z 3 csv súborov `units.csv`, `readings.csv`, `gateways.csv`

## Readings.csv

### 1.1A - Analýza štruktúr dát

#### Súbor

- **Počet záznamov:** 14495
- **Počet atribútov:** 11
- **Index:** `RangeIndex` 0 – 14494
- **Delimiter:** \t
- **Obsah:** merania senzorov z Air-Production-Unit (APU) vlaku, kde každý riadok je jedno meranie jednej jednotky `id_unit` v čase `ts_cycle`


#### Atribúty

| Atribút | Dtype | Non-null | Chýba | Popis |
|---|---|---|---|---|
| `id_unit` | str -> int64 | 14 495 | 0 | ID jednotky (APU) |
| `ts_cycle` | str -> datetime64 | 14 495 | 0 | Timestamp merania |
| `motor_current` | float64 | 14 495 | 0 | Prúd motora kompresora |
| `dpFilter` | float64 | 13 935 | 560 | Diferenčný tlak (podľa googlu) - indikuje zanášanie filtra |
| `temperature_oil` | float64 | 14 495 | 0 | Teplota oleja |
| `reservoirPressure` | float64 | 14 495 | 0 | Tlak v zásobníku |
| `oilPressure` | float64 | 14 495 | 0 | Tlak oleja |
| `rms_vibration` | float64 | 13 930 | 565 | Podľa wikipédie "Efektívna hodnota" vibrácií |
| `Flowmeter` | float64 | 14 495 | 0 | Prietok |
| `humidityIntake` | float64 | 13 790 | 705 | Vlhkosť nasávaného vzduchu |
| `failure` | int64 | 14 495 | 0 | Indikátor poruchy - predikovaná premenná. Teoreticky by to mohol byť `boolean`|


 Stĺpec *ts_cycle* obsahoval zmiešané formáty dátumov a časov, takže sme ich zjednotili pomocou parametra `parse_dates` vo funkcii `read_csv`. Okrem toho sme chceli zjednotiť ID APU *id_unit* medzi súbormi, tak sme ho prekonvertovali na `int` po odstránení "U" zo začiatku stringu.


In [54]:
READINGS = "readings.csv"
readings_df = pd.read_csv(READINGS, delimiter="\t", parse_dates=["ts_cycle"], date_format="mixed")

readings_df["id_unit"] = readings_df["id_unit"].str.replace("U", "").astype(int)

readings_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 14495 entries, 0 to 14494
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   id_unit            14495 non-null  int64         
 1   ts_cycle           14495 non-null  datetime64[us]
 2   motor_current      14495 non-null  float64       
 3   dpFilter           13935 non-null  float64       
 4   temperature_oil    14495 non-null  float64       
 5   reservoirPressure  14495 non-null  float64       
 6   oilPressure        14495 non-null  float64       
 7   rms_vibration      13930 non-null  float64       
 8   Flowmeter          14495 non-null  float64       
 9   humidityIntake     13790 non-null  float64       
 10  failure            14495 non-null  int64         
dtypes: datetime64[us](1), float64(8), int64(2)
memory usage: 1.2 MB


## Units.csv

### 1.1A - Analýza štruktúr dát

#### Súbor

- **Počet záznamov:** 2000
- **Počet atribútov:** 7
- **Index:** `RangeIndex` 0 – 1999
- **Delimiter:** ,
- **Obsah:** údaje o jednotkách APU. Nemenia sa v čase, tak uvidíme ešte ako relevantné pre nás budú `year_commision`, `dutyClass` alebo `hours_service`

#### Atribúty

| Atribút | Dtype | Non-null | Chýba | Popis |
|---|---|---|---|---|
| `id_unit` | int64 | 2 000 | 0 | Identifikátor jednotky APU, prepojenie so súborom `readings.csv` |
| `serial_number` | str | 2 000 | 0 | Sériové číslo jednotky |
| `year_commission` | int64 | 2 000 | 0 | Rok uvedenia do prevádzky |
| `dutyClass` | str | 2 000 | 0 | Trieda zaťaženia - kategorická premenná |
| `hours_service` | float64 | 1 900 | 100 | Počet prevádzkových hodín |
| `operatorOwner` | str | 2 000 | 0 | Prevádzkovateľ / vlastník |
| `idDepot` | int64 | 2 000 | 0 | Identifikátor depa |

Stĺpec *dutyClass* obsahuje rôzne pomenovania pre pravdepodobne rovnaké kategórie, no zatiaľ nevieme ktoré môžme zlúčiť.

In [55]:
UNITS = "units.csv"
units_df = pd.read_csv(UNITS, delimiter=",")

units_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id_unit          2000 non-null   int64  
 1   serial_number    2000 non-null   str    
 2   year_commission  2000 non-null   int64  
 3   dutyClass        2000 non-null   str    
 4   hours_service    1900 non-null   float64
 5   operatorOwner    2000 non-null   str    
 6   idDepot          2000 non-null   int64  
dtypes: float64(1), int64(3), str(3)
memory usage: 167.2 KB


## Gateways.csv

### 1.1A - Analýza štruktúr dát

#### Súbor

- **Počet záznamov:** 2000
- **Počet atribútov:** 14
- **Index:** `RangeIndex` 0 – 1999
- **Delimiter:** ;
- **Obsah:** údaje o komunikačných bránach (routeroch). Myslíme si, že tieto dáta budú skoro úplne nepodstatné pre naše predikcie

#### Atribúty

| Atribút | Dtype | Non-null | Chýba | Popis |
|---|---|---|---|---|
| `gatewayId` | str | 2 000 | 0 | Identifikátor brány |
| `id_unit` | float64 -> int64 | 2 000 | 0 | ID jednotky APU, ku ktorej brána patrí - foreign key|
| `idDepot` | int64 | 2 000 | 0 | Identifikátor depa |
| `lat` | float64 | 2 000 | 0 | Zemepisná šírka |
| `Lon` | float64 | 2 000 | 0 | Zemepisná dĺžka |
| `model_hardware` | str | 2 000 | 0 | Model hardvéru |
| `version_firmware` | str | 2 000 | 0 | Verzia firmvéru |
| `vendor` | str | 2 000 | 0 | Výrobca / dodávateľ |
| `Technician` | str | 2 000 | 0 | Technik, ktorý bránu inštaloval |
| `mac_address` | str | 2 000 | 0 | MAC adresa brány |
| `Ipv4` | str | 2 000 | 0 | IPv4 adresa brány |
| `dbm_signal` | float64 | 2 000 | 0 | Sila signálu v dBm |
| `dateInstall` | str -> datetime64 | 2 000 | 0 | Dátum inštalácie brány |
| `last_seen` | str -> datetime64 | 2 000 | 0 | Posledné zaznamenané spojenie brány |


Stĺpce *last_seen* a *dateInstall* oba používajú nejednotné formáty času, tak sme sa ich rozhodli zjednotiť.
Z povahy dát usudzujeme, že stĺpec *id_unit* bude značiť ID jednotky APU, do ktorej bola daný smerovač osadený. Ak smerovač ID APU nemá, znamená to že ho nevieme k žiadnemu priradiť a je pre nás irelevantný -> `dropna`. 
Stĺpec *id_unit* bol uložený ako typ `float`, tak sme ho prekonvertovali na typ `int`.

In [56]:
GATEWAYS = "gateways.csv"
gateways_df = pd.read_csv(GATEWAYS, delimiter=";", parse_dates=["last_seen", "dateInstall"], date_format="mixed")

gateways_df = gateways_df.dropna(subset="id_unit")
gateways_df["id_unit"] = gateways_df.id_unit.astype(int)

gateways_df.info()


<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   gatewayId         2000 non-null   str           
 1   lat               2000 non-null   float64       
 2   model_hardware    2000 non-null   str           
 3   version_firmware  2000 non-null   str           
 4   Technician        2000 non-null   str           
 5   mac_address       2000 non-null   str           
 6   idDepot           2000 non-null   int64         
 7   id_unit           2000 non-null   int64         
 8   vendor            2000 non-null   str           
 9   Lon               2000 non-null   float64       
 10  Ipv4              2000 non-null   str           
 11  last_seen         2000 non-null   datetime64[us]
 12  dbm_signal        2000 non-null   float64       
 13  dateInstall       2000 non-null   datetime64[us]
dtypes: datetime64[us](2), float64(3), i